In this notebook, we will use the CodeFlare SDK to:
 - Create a Ray cluster with in-tree autoscaling enabled
 - Observe workers scale up under load and scale down when idle
 - Tear down the cluster when finished

Autoscaling is configured on `ClusterConfiguration` with `enable_autoscaling=True`, `min_workers`, and `max_workers`.

> **Important:** Ray cluster autoscaling is **not supported when Kueue manages your workloads**. If Kueue is enabled for your project, the SDK raises an error when autoscaling is requested. Support for elastic Ray jobs with Kueue is tracked in [RHAIRFE-909](https://redhat.atlassian.net/browse/RHAIRFE-909).

In [ ]:
import os
import subprocess
import time

from kubernetes import client as k8s_client

from codeflare_sdk import Cluster, ClusterConfiguration, set_api_client
from codeflare_sdk.common.kubernetes_cluster import get_api_client
from kube_authkit import AuthConfig, get_k8s_client

In [ ]:
# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches the workbench service account may not have Ray RBAC
# permissions. Use Option 2 (token) unless your admin has granted SA permissions
# (see RHOAIENG-46748). Auto-detect works if you have a local kubeconfig.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,  # Interactive device flow for notebook environments
# )

api_client = get_k8s_client(config=auth_config)
set_api_client(api_client)

Define an autoscaling Ray cluster. With `enable_autoscaling=True`, the SDK sets `enableInTreeAutoscaling` on the RayCluster and maps `min_workers` / `max_workers` to worker replica bounds.

Use a **non-Kueue** namespace (no default local queue that routes RayClusters through Kueue).

NOTE: The default Ray images depend on the installed Python version:

- For Python 3.11: `quay.io/modh/ray:2.52.1-py311-cu121`
- For Python 3.12: `quay.io/modh/ray:2.55.1-py312-cu128`

In [ ]:
NAMESPACE = "rhods-notebooks"  # Use your data science project namespace
CLUSTER_NAME = "autoscale-demo"
MIN_WORKERS = 1
MAX_WORKERS = 2

cluster = Cluster(
    ClusterConfiguration(
        name=CLUSTER_NAME,
        namespace=NAMESPACE,
        enable_autoscaling=True,
        min_workers=MIN_WORKERS,
        max_workers=MAX_WORKERS,
        head_cpu_requests=1,
        head_cpu_limits=1,
        head_memory_requests=7,
        head_memory_limits=8,
        head_extended_resource_requests={"nvidia.com/gpu": 0},
        worker_extended_resource_requests={"nvidia.com/gpu": 0},
        worker_cpu_requests=1,
        worker_cpu_limits=1,
        worker_memory_requests=5,
        worker_memory_limits=6,
        write_to_file=False,
        # verify_tls=False,  # Uncomment if your cluster uses self-signed API certs
    )
)

Create the Ray cluster.

In [ ]:
cluster.apply()

Wait until the cluster is ready, then review its details.

In [ ]:
cluster.wait_ready()
cluster.details()

The helper below counts running worker pods for this Ray cluster. After the cluster is ready, you should see `min_workers` workers.

In [ ]:
def count_workers(cluster_name: str, namespace: str) -> int:
    api = k8s_client.CoreV1Api(get_api_client())
    label = f"ray.io/node-type=worker,ray.io/cluster={cluster_name}"
    pods = api.list_namespaced_pod(namespace, label_selector=label)
    return len(pods.items or [])


def wait_for_worker_count(
    cluster_name: str,
    namespace: str,
    predicate,
    timeout_s: int = 900,
    poll_s: int = 10,
) -> int:
    start = time.time()
    last = None
    while time.time() - start < timeout_s:
        last = count_workers(cluster_name, namespace)
        print(f"Worker count: {last}")
        if predicate(last):
            return last
        time.sleep(poll_s)
    raise TimeoutError(
        f"Timed out waiting for worker count. cluster={cluster_name} last={last}"
    )


def _head_pod_name(cluster_name: str, namespace: str) -> str:
    api = k8s_client.CoreV1Api(get_api_client())
    label = f"ray.io/node-type=head,ray.io/cluster={cluster_name}"
    pods = api.list_namespaced_pod(namespace, label_selector=label)
    if not pods.items:
        raise RuntimeError(f"No head pod found for cluster {cluster_name}")
    return pods.items[0].metadata.name


def submit_autoscaling_load(
    cluster,
    cluster_name: str,
    namespace: str,
    tasks: int = 3,
    sleep_s: int = 180,
) -> dict:
    """Submit CPU load via Ray Jobs API or head-pod exec when dashboard is unavailable."""
    dashboard_uri = cluster.cluster_dashboard_uri()
    if dashboard_uri.startswith("http"):
        client = cluster.job_client
        submission_id = client.submit_job(
            entrypoint=(
                f"AUTOSCALING_TASKS={tasks} AUTOSCALING_TASK_SLEEP_S={sleep_s} "
                "python autoscaling_load.py"
            ),
            runtime_env={"working_dir": "./"},
        )
        print(f"Submitted autoscaling load job via dashboard: {submission_id}")
        print(f"Job status: {client.get_job_status(submission_id)}")
        return {"mode": "job_client", "client": client, "submission_id": submission_id}

    load_script = os.path.join(os.getcwd(), "autoscaling_load.py")
    head_pod = _head_pod_name(cluster_name, namespace)
    subprocess.check_call(
        [
            "kubectl",
            "cp",
            load_script,
            f"{namespace}/{head_pod}:/tmp/autoscaling_load.py",
        ]
    )
    proc = subprocess.Popen(
        [
            "kubectl",
            "exec",
            "-n",
            namespace,
            head_pod,
            "--",
            "bash",
            "-lc",
            (
                f"AUTOSCALING_TASKS={tasks} AUTOSCALING_TASK_SLEEP_S={sleep_s} "
                "python /tmp/autoscaling_load.py"
            ),
        ]
    )
    print(f"Started autoscaling load in head pod {head_pod} (dashboard unavailable)")
    return {"mode": "exec", "proc": proc}


def wait_for_autoscaling_load(load_handle: dict) -> None:
    if load_handle["mode"] == "job_client":
        client = load_handle["client"]
        submission_id = load_handle["submission_id"]
        while client.get_job_status(submission_id) in {"PENDING", "RUNNING"}:
            print(f"Job status: {client.get_job_status(submission_id)}")
            time.sleep(15)
        print(f"Final job status: {client.get_job_status(submission_id)}")
        print(client.get_job_logs(submission_id))
    else:
        rc = load_handle["proc"].wait(timeout=900)
        if rc != 0:
            raise RuntimeError(f"Head-pod load script failed with exit code {rc}")
        print("Head-pod load script finished successfully")


def cleanup_autoscaling_load(load_handle: dict) -> None:
    if load_handle["mode"] == "job_client":
        load_handle["client"].delete_job(load_handle["submission_id"])


initial_workers = wait_for_worker_count(
    CLUSTER_NAME, NAMESPACE, lambda n: n == MIN_WORKERS, timeout_s=600
)
print(f"Initial worker count matches min_workers={MIN_WORKERS}: {initial_workers}")

Submit a CPU-bound Ray job that queues more tasks than the cluster can run with the current worker count. With `max_workers=2`, three tasks at 1 CPU each should trigger a scale-up to two workers while the job runs.

The job script `autoscaling_load.py` is included alongside this notebook. When the Ray dashboard URL is available, the load is submitted through the Jobs API. Otherwise (for example on KinD without routes), the notebook runs the script in the head pod with `kubectl exec`.

In [ ]:
LOAD_TASKS = 3
LOAD_SLEEP_S = 180

load_handle = submit_autoscaling_load(
    cluster, CLUSTER_NAME, NAMESPACE, tasks=LOAD_TASKS, sleep_s=LOAD_SLEEP_S
)

In [ ]:
scaled_up = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n >= MAX_WORKERS,
    timeout_s=900,
)
print(f"Scale-up observed: {scaled_up} worker(s) (max_workers={MAX_WORKERS})")

Wait for the load job to finish, then wait for the autoscaler to scale back down to `min_workers`. Scale-down can take several minutes after the cluster becomes idle.

In [ ]:
wait_for_autoscaling_load(load_handle)

scaled_down = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n == MIN_WORKERS,
    timeout_s=900,
)
print(f"Scale-down observed: {scaled_down} worker(s) (min_workers={MIN_WORKERS})")

Delete the submitted job and tear down the Ray cluster.

In [ ]:
cleanup_autoscaling_load(load_handle)
cluster.down()

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit